# EXHEART 13 — Operating characteristics of the cross-instrument disparity attenuation ratio

**What this establishes.** The manuscript's proof-of-concept simulation shows the disparity attenuation ratio *R* = (Δ_subj − Δ_obj)/Δ_subj responds sensibly when a gap is purely measurement-induced versus purely substantive. That is face validity under two hand-picked regimes. It does **not** characterise the statistic as a diagnostic, and in particular it never tests the confound a reviewer will raise: **outcome-definition mismatch**, where the two instruments measure different-but-related outcomes with no measurement bias at all.

This notebook runs a full simulation grid and reports, against a **predeclared decision rule** (call an attenuation *measurement-induced* if R̂ ≥ 0.5), the statistic's:
- **discrimination** (ROC-AUC of R̂) between mechanisms,
- **sensitivity / specificity** of the rule,
- **false-positive rate under outcome-definition mismatch** — the key test,
- **bias and RMSE** of R̂ against its ground-truth value,
- **bootstrap-CI coverage**,
- **robustness** to prevalence, threshold rule, sample size, and model family.

The honest hypothesis, which the notebook will confirm or refute: R separates *instrument-related* attenuation from *substantive* difference, but **cannot identify the channel** — it flags outcome-definition mismatch as "measurement-induced" just as readily as true measurement bias. If so, that is the real, publishable contribution: cross-instrument disparity comparison requires outcome equivalence, or it conflates two mechanisms. Results to `results/diagnostic_characterization/`.

### 1. Determinism

In [1]:
import os
os.environ['PYTHONHASHSEED']='42'
import random, numpy as np
random.seed(42); np.random.seed(42)

### 2. Mount, git identity, paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, json, itertools
import pandas as pd, matplotlib.pyplot as plt
DRIVE_ROOT='/content/drive/MyDrive/EXHEART_Research'; REPO_DIR=os.path.join(DRIVE_ROOT,'exheart-research')
for fn in ['.git-credentials','.gitconfig']:
    s=os.path.join(DRIVE_ROOT,fn)
    if os.path.exists(s): shutil.copy(s, os.path.join('/root',fn)); print('restored',fn)
RES=os.path.join(REPO_DIR,'results/diagnostic_characterization'); os.makedirs(RES+'/figures',exist_ok=True); os.makedirs(RES+'/tables',exist_ok=True)
print('ready')

Mounted at /content/drive
ready


### 3. Libraries

In [3]:
!pip install -q scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

### 4. Data-generating process

One population; protected attribute A (female = 1, balanced); four observed features X plus A as a model feature (Sex is a feature in EXHEART); an unobserved risk component U used to create genuine, hard-to-detect group differences. Each regime returns a **subjective** instrument (features, label) and an **objective** instrument (features, label). Both classifiers are trained on the observed features and A only.

- **null** — no disparity anywhere (negative control).
- **measurement** — clean outcome; the *subjective* label under-reports a fraction *s* of female positives (label-measurement bias, as with self-reported CVD in women). Objective label clean. Expect Δ_subj>0, Δ_obj≈0 ⇒ R→1.
- **substantive** — both instruments measure the *same* outcome, in which female risk depends partly on unobserved U (genuinely harder to detect). Gap persists on both ⇒ R→0.
- **mixed** — measurement + substantive combined ⇒ R intermediate.
- **outcome_mismatch** — *no measurement bias*; the subjective instrument measures outcome Y1 (group-associated via U1) and the objective instrument measures a *different* outcome Y2 with no group association. Gap attenuates purely because the outcomes differ ⇒ R→1, **mimicking measurement**. This is the confound.
- **symmetric** — label noise applied equally to both groups (control): no group gap ⇒ diagnostic should not trigger.

In [4]:
def make_pop(n, prevalence, rng):
    A=rng.integers(0,2,n)
    X=rng.normal(size=(n,4))
    U=rng.normal(size=n)
    eta=0.9*X[:,0]+0.7*X[:,1]+0.5*X[:,2]+0.4*X[:,3]
    # intercept for target prevalence on the base (no-group) outcome
    b0=np.quantile(-eta, 1-prevalence)
    return A,X,U,eta,b0

def bern(p,rng): return (rng.random(len(p))<p).astype(int)
def sig(z): return 1/(1+np.exp(-z))

def gen_regime(regime, s, n, prevalence, rng):
    """Return (Xf, A, y_subj, y_obj): features incl. A, and the two instrument labels."""
    A,X,U,eta,b0=make_pop(n,prevalence,rng)
    Xf=np.column_stack([X, A])                     # A is a model feature
    base=sig(b0+eta)
    if regime=='null':
        y=bern(base,rng); ys=y.copy(); yo=y.copy()
    elif regime=='measurement':
        y=bern(base,rng); yo=y.copy(); ys=y.copy()
        fempos=(A==1)&(ys==1)
        flip=fempos&(rng.random(n)<s)              # under-report a fraction s of female positives
        ys[flip]=0
    elif regime=='symmetric':
        y=bern(base,rng); yo=y.copy(); ys=y.copy()
        pos=(ys==1); flip=pos&(rng.random(n)<s)    # same flip rate for both groups
        ys[flip]=0
    elif regime=='substantive':
        eta_t=b0+eta+s*A*U                          # female outcome depends on unobserved U
        y=bern(sig(eta_t),rng); ys=y.copy(); yo=y.copy()
    elif regime=='mixed':
        eta_t=b0+eta+ (0.6*s)*A*U
        y=bern(sig(eta_t),rng); yo=y.copy(); ys=y.copy()
        fempos=(A==1)&(ys==1); flip=fempos&(rng.random(n)< (0.6*s if s<1 else 0.35))
        ys[flip]=0
    elif regime=='outcome_mismatch':
        # subjective measures Y1 (group-associated via U1); objective measures a DIFFERENT outcome Y2 (no group assoc)
        U1=U
        ys=bern(sig(b0+eta+s*A*U1),rng)            # group gap, no measurement bias
        yo=bern(sig(b0+eta),rng)                    # different outcome, no group association
    else:
        raise ValueError(regime)
    return Xf.astype(float), A, ys, yo

### 5. Fit both instruments, compute the group TPR gap and R

In [5]:
def mk_model(name):
    if name=='logistic': return LogisticRegression(max_iter=1000)
    if name=='hgb': return HistGradientBoostingClassifier(max_depth=4, max_iter=150, random_state=0)
    if name=='rf': return RandomForestClassifier(n_estimators=120, max_depth=8, random_state=0, n_jobs=-1)

def thr(scores, y, rule):
    if rule=='fixed': return 0.5
    if rule=='prevmatch': return np.quantile(scores, 1-max(y.mean(),1e-3))
    if rule=='youden':
        from sklearn.metrics import roc_curve
        fpr,tpr,t=roc_curve(y,scores); return t[np.argmax(tpr-fpr)]

def tpr_gap(scores, y, A, t):
    def tpr(mask):
        pos=mask&(y==1)
        return (scores[pos]>=t).mean() if pos.sum()>0 else np.nan
    return tpr(A==1)-tpr(A==0)   # female minus male (signed); magnitude used for R

def eval_once(regime, s, n, prevalence, model, rule, rng, eps=0.02):
    Xf,A,ys,yo=gen_regime(regime,s,n,prevalence,rng)
    out={}
    for lab,y in [('subj',ys),('obj',yo)]:
        Xtr,Xte,ytr,yte,Atr,Ate=train_test_split(Xf,y,A,test_size=0.4,random_state=int(rng.integers(1e9)),stratify=y)
        m=mk_model(model); m.fit(Xtr,ytr); sc=m.predict_proba(Xte)[:,1]
        t=thr(sc,yte,rule); out[lab]=abs(tpr_gap(sc,yte,Ate,t))
    gs,go=out['subj'],out['obj']
    triggered = gs>=eps
    R = (gs-go)/gs if gs>eps else np.nan
    return gs,go,R,triggered

### 6. Self-check: one run per regime (sanity before the full grid)

In [6]:
rng=np.random.default_rng(1)
print(f"{'regime':16s} {'gap_subj':>9s} {'gap_obj':>9s} {'R':>7s}  triggered")
for reg in ['null','measurement','substantive','mixed','outcome_mismatch','symmetric']:
    s={'measurement':0.5,'symmetric':0.5,'substantive':1.8,'mixed':1.5,'outcome_mismatch':1.8,'null':0}[reg]
    gs,go,R,tr=eval_once(reg,s,8000,0.30,'logistic','prevmatch',rng)
    print(f"{reg:16s} {gs:9.3f} {go:9.3f} {('nan' if R!=R else f'{R:6.3f}')}   {tr}")
print("\nExpected: measurement & outcome_mismatch -> R near 1; substantive -> R near 0;")
print("null & symmetric -> gap_subj ~0 (not triggered). If so, the DGP is behaving.")

regime            gap_subj   gap_obj       R  triggered
null                 0.036     0.030  0.158   True
measurement          0.648     0.038  0.941   True
substantive          0.058     0.112 -0.941   True
mixed                0.492     0.017  0.965   True
outcome_mismatch     0.103     0.018  0.822   True
symmetric            0.015     0.031 nan   False

Expected: measurement & outcome_mismatch -> R near 1; substantive -> R near 0;
null & symmetric -> gap_subj ~0 (not triggered). If so, the DGP is behaving.


### 7. Main grid — mechanism × effect strength × model (Monte Carlo)

In [7]:
N=6000; PREV=0.30; RULE='prevmatch'; N_REP=100
STRENGTH={'measurement':[0.25,0.45,0.65],'symmetric':[0.25,0.45,0.65],
          'substantive':[0.9,1.7,2.6],'mixed':[0.9,1.7,2.6],
          'outcome_mismatch':[0.9,1.7,2.6],'null':[0.0]}
MODELS=['logistic','hgb']
rows=[]
for model in MODELS:
    for reg,strengths in STRENGTH.items():
        for si,s in enumerate(strengths):
            rr=np.random.default_rng(1000+hash((model,reg,si))%99999)
            for rep in range(N_REP):
                gs,go,R,tr=eval_once(reg,s,N,PREV,model,RULE,rr)
                rows.append(dict(model=model,regime=reg,strength=['low','med','high'][si] if len(strengths)>1 else 'na',
                                 gap_subj=gs,gap_obj=go,R=R,triggered=bool(tr)))
        print(f'done {model}/{reg}')
main=pd.DataFrame(rows); main.to_csv(RES+'/tables/main_grid_raw.csv',index=False)
print('rows:',len(main))

done logistic/measurement
done logistic/symmetric
done logistic/substantive
done logistic/mixed
done logistic/outcome_mismatch
done logistic/null
done hgb/measurement
done hgb/symmetric
done hgb/substantive
done hgb/mixed
done hgb/outcome_mismatch
done hgb/null
rows: 3200


### 8. Operating characteristics against the predeclared rule (R̂ ≥ 0.5)

In [8]:
TAU=0.5
def summarize(df):
    g=df.dropna(subset=['R'])
    s=g.groupby(['model','regime','strength'])['R'].agg(['mean','std','count'])
    s['pct_triggered']=df.groupby(['model','regime','strength'])['triggered'].mean()
    s['pct_flag_measurement']=g.assign(flag=g.R>=TAU).groupby(['model','regime','strength'])['flag'].mean()
    return s.round(3)
summ=summarize(main); summ.to_csv(RES+'/tables/regime_summary.csv'); print(summ)

                                     mean    std  count  pct_triggered  \
model    regime           strength                                       
hgb      measurement      high      0.974  0.018    100           1.00   
                          low       0.918  0.069    100           1.00   
                          med       0.963  0.026    100           1.00   
         mixed            high      0.869  0.065    100           1.00   
                          low       0.967  0.027    100           1.00   
                          med       0.917  0.059    100           1.00   
         null             na        0.263  0.594     44           0.44   
         outcome_mismatch high      0.781  0.193    100           1.00   
                          low       0.384  0.492     57           0.57   
                          med       0.683  0.323     93           0.93   
         substantive      high     -0.019  0.397     99           0.99   
                          low       0.

In [9]:
# Discrimination: can R separate mechanisms? (logistic, med strength, triggered reps)
def auc_between(df, regA, regB, model='logistic', strength='med'):
    a=df[(df.model==model)&(df.regime==regA)&(df.strength.isin([strength,'na']))].dropna(subset=['R'])
    b=df[(df.model==model)&(df.regime==regB)&(df.strength.isin([strength,'na']))].dropna(subset=['R'])
    if len(a)<5 or len(b)<5: return np.nan
    y=np.r_[np.ones(len(a)),np.zeros(len(b))]; sc=np.r_[a.R.values,b.R.values]
    return roc_auc_score(y,sc)
print('AUC of R separating mechanisms (1.0 = perfect, 0.5 = cannot separate):')
print(f"  measurement vs substantive     : {auc_between(main,'measurement','substantive'):.3f}   (face validity)")
print(f"  measurement vs outcome_mismatch: {auc_between(main,'measurement','outcome_mismatch'):.3f}   (THE KEY TEST)")
print(f"  {'{measurement,mismatch} vs substantive':38s}:", end=' ')
inst=main[(main.model=='logistic')&(main.regime.isin(['measurement','outcome_mismatch']))&(main.strength=='med')].dropna(subset=['R'])
sub =main[(main.model=='logistic')&(main.regime=='substantive')&(main.strength=='med')].dropna(subset=['R'])
y=np.r_[np.ones(len(inst)),np.zeros(len(sub))]; print(f"{roc_auc_score(y,np.r_[inst.R.values,sub.R.values]):.3f}   (instrument-related vs substantive)")

AUC of R separating mechanisms (1.0 = perfect, 0.5 = cannot separate):
  measurement vs substantive     : 0.999   (face validity)
  measurement vs outcome_mismatch: 0.851   (THE KEY TEST)
  {measurement,mismatch} vs substantive : 0.965   (instrument-related vs substantive)


In [10]:
# Sensitivity / specificity of the R>=0.5 rule, and the false-positive rate under outcome mismatch
def rate_flag(reg, model='logistic', strength='med'):
    d=main[(main.model==model)&(main.regime==reg)&(main.strength.isin([strength,'na']))].dropna(subset=['R'])
    return (d.R>=TAU).mean(), len(d)
sens,_=rate_flag('measurement'); spec_sub,_=rate_flag('substantive'); fpr_mis,_=rate_flag('outcome_mismatch')
print('Predeclared rule  R̂ >= 0.5  ->  call "measurement-induced":')
print(f"  sensitivity (flag | true measurement)      : {sens:.3f}")
print(f"  specificity (not flag | true substantive)  : {1-spec_sub:.3f}")
print(f"  FALSE POSITIVE (flag | outcome mismatch)   : {fpr_mis:.3f}   <- if high, R cannot identify the channel")

Predeclared rule  R̂ >= 0.5  ->  call "measurement-induced":
  sensitivity (flag | true measurement)      : 1.000
  specificity (not flag | true substantive)  : 0.918
  FALSE POSITIVE (flag | outcome mismatch)   : 0.792   <- if high, R cannot identify the channel


### 9. Bias and RMSE of R̂ vs ground-truth value (R_true: measurement=1, substantive=0)

In [11]:
def bias_rmse(reg, Rtrue, model='logistic', strength='med'):
    d=main[(main.model==model)&(main.regime==reg)&(main.strength.isin([strength,'na']))].dropna(subset=['R'])
    err=d.R.values-Rtrue; return err.mean(), np.sqrt((err**2).mean()), len(d)
for reg,rt in [('measurement',1.0),('substantive',0.0)]:
    b,r,n=bias_rmse(reg,rt); print(f"  {reg:16s} R_true={rt}:  bias={b:+.3f}  RMSE={r:.3f}  (n={n})")

  measurement      R_true=1.0:  bias=-0.046  RMSE=0.056  (n=100)
  substantive      R_true=0.0:  bias=-0.146  RMSE=0.675  (n=97)


### 10. Bootstrap-CI coverage (does the paper's 95% CI cover R_true?)

In [12]:
def eval_with_ci(regime,s,n,prevalence,rng,B=200,eps=0.02):
    Xf,A,ys,yo=gen_regime(regime,s,n,prevalence,rng)
    cache={}
    for lab,y in [('subj',ys),('obj',yo)]:
        Xtr,Xte,ytr,yte,Atr,Ate=train_test_split(Xf,y,A,test_size=0.4,random_state=int(rng.integers(1e9)),stratify=y)
        m=mk_model('logistic').fit(Xtr,ytr); sc=m.predict_proba(Xte)[:,1]; t=thr(sc,yte,'prevmatch')
        cache[lab]=(sc,yte,Ate,t)
    def R_of(idx_s,idx_o):
        sc,y,Aa,t=cache['subj']; gs=abs(tpr_gap(sc[idx_s],y[idx_s],Aa[idx_s],t))
        sc,y,Aa,t=cache['obj'];  go=abs(tpr_gap(sc[idx_o],y[idx_o],Aa[idx_o],t))
        return (gs-go)/gs if gs>eps else np.nan
    ns=len(cache['subj'][1]); no=len(cache['obj'][1])
    Rs=[R_of(rng.integers(0,ns,ns),rng.integers(0,no,no)) for _ in range(B)]
    Rs=np.array([r for r in Rs if r==r])
    return (np.nanpercentile(Rs,2.5), np.nanpercentile(Rs,97.5)) if len(Rs)>10 else (np.nan,np.nan)

for reg,rt,s in [('measurement',1.0,0.45),('substantive',0.0,1.7)]:
    rng=np.random.default_rng(77); cov=0; tot=0
    for _ in range(100):
        lo,hi=eval_with_ci(reg,s,6000,0.30,rng)
        if lo==lo: tot+=1; cov+= (lo<=rt<=hi)
    print(f"  {reg:16s} R_true={rt}: 95% CI coverage = {cov/max(tot,1):.2f}  (nominal 0.95, n={tot})")

  measurement      R_true=1.0: 95% CI coverage = 0.00  (nominal 0.95, n=100)
  substantive      R_true=0.0: 95% CI coverage = 0.88  (nominal 0.95, n=100)


### 11. Robustness — prevalence, threshold rule, sample size, model (med strength)

In [ ]:
def sep_auc(model,rule,prev,n,strength_meas=0.45,strength_sub=1.7,strength_mis=1.7,nrep=60):
    rr=np.random.default_rng(9)
    R={'measurement':[],'substantive':[],'outcome_mismatch':[]}
    smap={'measurement':strength_meas,'substantive':strength_sub,'outcome_mismatch':strength_mis}
    for reg in R:
        for _ in range(nrep):
            _,_,r,_=eval_once(reg,smap[reg],n,prev,model,rule,rr)
            if r==r: R[reg].append(r)
    def auc(a,b):
        if len(a)<5 or len(b)<5: return np.nan
        return roc_auc_score(np.r_[np.ones(len(a)),np.zeros(len(b))], np.r_[a,b])
    return auc(R['measurement'],R['substantive']), auc(R['measurement'],R['outcome_mismatch'])
robust=[]
for prev in [0.1,0.3,0.5]:
    a1,a2=sep_auc('logistic','prevmatch',prev,6000); robust.append(('prevalence',prev,a1,a2))
for rule in ['fixed','prevmatch','youden']:
    a1,a2=sep_auc('logistic',rule,0.30,6000); robust.append(('threshold',rule,a1,a2))
for n in [2000,6000,20000]:
    a1,a2=sep_auc('logistic','prevmatch',0.30,n); robust.append(('sample_size',n,a1,a2))
for mdl in ['logistic','hgb','rf']:
    a1,a2=sep_auc(mdl,'prevmatch',0.30,6000); robust.append(('model',mdl,a1,a2))
rob=pd.DataFrame(robust,columns=['factor','value','AUC_meas_vs_subst','AUC_meas_vs_mismatch']).round(3)
rob.to_csv(RES+'/tables/robustness.csv',index=False); print(rob.to_string(index=False))

### 12. Figures

In [ ]:
g=main[(main.model=='logistic')].dropna(subset=['R'])
fig,ax=plt.subplots(1,2,figsize=(13,4.8))
order=['substantive','mixed','measurement','outcome_mismatch']
data=[g[(g.regime==r)&(g.strength.isin(['med','na']))].R.values for r in order]
ax[0].boxplot(data,labels=['substantive','mixed','measurement','outcome\nmismatch'],showmeans=True)
ax[0].axhline(0.5,ls=':',c='crimson'); ax[0].text(0.6,0.52,'decision rule R=0.5',color='crimson',fontsize=8)
ax[0].set_ylabel('disparity attenuation ratio  R̂'); ax[0].set_title('R by mechanism (med strength, logistic)')
ax[0].annotate('measurement and outcome-mismatch\nare indistinguishable',xy=(3.5,0.9),xytext=(2.0,0.15),
               fontsize=8,color='crimson',arrowprops=dict(arrowstyle='->',color='crimson'))
rob_m=rob[rob.factor=='model']
x=np.arange(len(rob_m)); w=0.35
ax[1].bar(x-w/2,rob_m.AUC_meas_vs_subst,w,label='measurement vs substantive',color='seagreen')
ax[1].bar(x+w/2,rob_m.AUC_meas_vs_mismatch,w,label='measurement vs outcome-mismatch',color='indianred')
ax[1].axhline(0.5,ls=':',c='grey'); ax[1].set_xticks(x); ax[1].set_xticklabels(rob_m.value)
ax[1].set_ylim(0,1.05); ax[1].set_ylabel('AUC of R̂'); ax[1].set_title('Discrimination by model family'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RES+'/figures/diagnostic_characterization.png',dpi=150,bbox_inches='tight'); plt.show()

### 13. Verdict

In [ ]:
auc_ms=auc_between(main,'measurement','substantive')
auc_mm=auc_between(main,'measurement','outcome_mismatch')
print('='*66); print('CHARACTERIZATION VERDICT'); print('='*66)
print(f"R separates measurement from substantive : AUC {auc_ms:.3f}")
print(f"R separates measurement from outcome-mism: AUC {auc_mm:.3f}")
if auc_ms>0.85 and auc_mm<0.65:
    print("\nFINDING (as hypothesised): R reliably distinguishes instrument-related")
    print("attenuation from substantive difference, but CANNOT identify the channel:")
    print("it flags outcome-definition mismatch as 'measurement-induced' as readily as")
    print("true measurement bias. Cross-instrument disparity comparison therefore")
    print("requires establishing outcome equivalence; without it, R conflates two")
    print("mechanisms. This is the contribution: a characterised statistic with a")
    print("stated operating range and a documented failure mode.")
else:
    print("\nResults differ from the hypothesis; interpret directly from the tables/figure")
    print("above rather than from a preset narrative.")
print('='*66)

### 14. Commit

In [ ]:
%cd {REPO_DIR}
!git add results/diagnostic_characterization notebooks/EXHEART_13_diagnostic_operating_characteristics.ipynb 2>/dev/null
!git commit -m "Add operating-characteristics study of the disparity attenuation ratio (incl. outcome-mismatch failure mode)"
!git push origin main